# 01 GPU — CIFAR-100 with GPU Support

## GPU Setup in Google Colab

To enable GPU acceleration in Google Colab, follow these steps:

1. Click **Runtime** in the top menu bar
2. Select **Change runtime type**
3. Under **Hardware accelerator**, choose **T4 GPU** (or another GPU option)
4. Click **Save**
5. The runtime will restart — re-run all cells from the beginning

> ⚠️ If you skip this step, PyTorch will fall back to CPU and training will be much slower.

In [ ]:
# ── GPU Verification ──────────────────────────────────────────────────────────
# Check GPU hardware info using the NVIDIA System Management Interface (nvidia-smi).
# If GPU is enabled, you will see the GPU model, memory, driver version, etc.
# If you see an error like 'nvidia-smi: not found', the runtime is using CPU only.
!nvidia-smi

print()

# Check whether PyTorch can detect and use the GPU via CUDA.
import torch
print("CUDA available :", torch.cuda.is_available())

# If CUDA is available, print the GPU model name.
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
    print("CUDA version   :", torch.version.cuda)
else:
    print("No GPU detected. Please enable GPU via Runtime > Change runtime type.")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH="/content/drive/MyDrive/ML-Basic/cifar100"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls *.py

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile 01gpu.py
# Adapted for CIFAR-100 with GPU support

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import torch.backends.cudnn as cudnn

import time
# Use GPU if available, otherwise fall back to CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size = 32
epochs = 10

# training data preparation
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

# CIFAR-100 has 100 classes (20 superclasses x 5 subclasses each)

# network definition
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        '''
            documents:
                Conv2d https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
                MaxPool2d https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html
                Flatten https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html
                Linear https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
                relu https://pytorch.org/docs/stable/generated/torch.nn.functional.relu.html
        '''

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1, padding_mode='replicate')
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1, padding_mode='zeros')

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(in_features=32 * 8 * 8, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=64)
        self.fc3 = nn.Linear(in_features=64, out_features=100)  # 100 classes

    def forward(self, x):
        x = self.conv1(x) # (B,3,32,32) -> (B,16,32,32)
        x = F.relu(x)
        x = self.pool(x) # (B,16,32,32) -> (B,16,16,16)

        x = self.conv2(x) # (B,16,16,16) -> (B,32,16,16)
        x = F.relu(x)
        x = self.pool(x) # (B,32,16,16) -> (B,32,8,8)

        x = self.flatten(x) # (B,32,8,8) -> (B,32*8*8)

        x = self.fc1(x) # (B,32*8*8) -> (B,128)
        x = F.relu(x)

        x = self.fc2(x) # (B,128) -> (B,64)
        x = F.relu(x)

        x = self.fc3(x) # (B,64) -> (B,100)

        return x

net = Net()
net = net.to(device)
if device == 'cuda':
    cudnn.benchmark = True
    print('Run with GPU')
else:
    print('Run with CPU')

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

net.train()
t0 = time.perf_counter()
for epoch in range(epochs):

    running_loss = 0.0
    for i, data in enumerate(trainloader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 100 == 99:
            t1 = time.perf_counter()
            print('[%d, %5d] loss: %.3f, time: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000, t1-t0))
            running_loss = 0.0
            t0 = t1

print('Finished Training')

net.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %.3f %%' % (100 * correct / total))

Writing 01gpu.py


In [ ]:
# Execute a Python file
%cd "{PROJECT_PATH}"
!python 01gpu.py

[![Return to GitHub](https://img.shields.io/badge/GitHub-Return-black?logo=github)](https://github.com/mastnk/practical-ml)